*italicized text*# Multi-Commodity Flow for Traffic Routing on Road Networks

## Problem Overview

**Scenario:** Multiple drivers need to travel from different origins to different destinations through a shared road network. Roads have limited capacity (number of lanes, traffic congestion limits).

**Challenge:** How do we route all vehicles to maximize total traffic flow while respecting road capacity constraints?

This is the **Multi-Commodity Flow (MCF) problem**: routing multiple different traffic flows (commodities) through a shared road network where roads have capacity limits.

## Real-World Applications

- **Urban traffic management**: Route vehicles to minimize congestion
- **Emergency evacuation**: Multiple evacuation routes from different neighborhoods
- **Delivery logistics**: Multiple delivery trucks with different destinations
- **Public transportation**: Bus routes serving different origin-destination pairs

## Dataset

**Source:** Stanford Large Network Dataset Collection (SNAP)

**Primary Choice: `roadNet-CA` (California Road Network)**
- **Nodes**: 1,965,206 intersections
- **Edges**: 2,766,607 road segments
- **Type**: Undirected (roads are bidirectional)
- **Download**: http://snap.stanford.edu/data/roadNet-CA.html

**For experiments:** Subsample to 1,000-5,000 nodes for tractability

**Alternatives:**
- `roadNet-PA`: Pennsylvania (1,088,092 nodes)
- `roadNet-TX`: Texas (1,379,917 nodes)

## Problem Interpretation

**Network components:**
- **Nodes** = Road intersections
- **Edges** = Road segments connecting intersections
- **Capacity** = Maximum vehicles per hour on each road
- **Commodities** = Different origin-destination traffic pairs (e.g., people going from home to work)
- **Demand** = Number of vehicles needing to travel from origin to destination

**Example:**
- Commodity 1: 100 cars from intersection A to intersection B (morning commute)
- Commodity 2: 50 cars from intersection C to intersection D (delivery trucks)
- ...
- All sharing the same road network with capacity constraints

## Mathematical Formulation

### Given
- Road network $G = (V, E)$
- Capacity $u_{ij}$ on each road segment $(i,j)$ (max vehicles/hour)
- $K$ commodities (traffic flows), each with:
  - Origin intersection $s_k$
  - Destination intersection $t_k$  
  - Demand $d_k$ (number of vehicles)

### Decision Variables
$x_{ij}^k$ = number of vehicles of commodity $k$ (traffic flow $k$) using road $(i,j)$

### Objective
Maximize total vehicles successfully routed:
$$\max \sum_{k=1}^K \sum_{i: (s_k, i) \in E} x_{s_k, i}^k$$

### Constraints

**1. Flow conservation** (at each intersection for each commodity):
$$\sum_{j: (i,j) \in E} x_{ij}^k - \sum_{j: (j,i) \in E} x_{ji}^k = \begin{cases}
d_k & \text{if } i = s_k \text{ (origin)} \\
-d_k & \text{if } i = t_k \text{ (destination)} \\
0 & \text{otherwise (intermediate intersection)}
\end{cases}$$

**2. Road capacity** (all traffic flows share road capacity):
$$\sum_{k=1}^K x_{ij}^k \leq u_{ij} \quad \forall (i,j) \in E$$

**3. Non-negativity**:
$$x_{ij}^k \geq 0 \quad \forall k, \forall (i,j)$$

### Complete Formulation
$$
\begin{align}
\max \quad & \sum_{k=1}^K \sum_{i: (s_k, i) \in E} x_{s_k, i}^k \\
\text{s.t.} \quad & \sum_{j: (i,j) \in E} x_{ij}^k - \sum_{j: (j,i) \in E} x_{ji}^k = b_i^k \quad \forall k, \forall i \in V \\
& \sum_{k=1}^K x_{ij}^k \leq u_{ij} \quad \forall (i,j) \in E \\
& x_{ij}^k \geq 0 \quad \forall k, \forall (i,j) \in E
\end{align}
$$

## Methods to Compare

### Baseline 1: Greedy Shortest Path
- Route each commodity on its shortest path sequentially
- Fast but ignores global capacity constraints
- **Uses what we learned**: Dijkstra's shortest path algorithm

### Baseline 2: Sequential Max Flow  
- Apply max flow algorithm for each commodity one-by-one
- Update residual capacities after each commodity
- **Uses what we learned**: Edmonds-Karp / Ford-Fulkerson max flow

### Baseline 3: Gurobi IP Solver
- Solve the full IP formulation directly
- Optimal but may be slow for large instances

### New Method: Column Generation
- **Path-based formulation**: Instead of edge flows, use path variables
- **Master problem**: Select which paths to use for each commodity
- **Pricing subproblem**: Find new profitable paths (shortest path with modified costs)
- **Why it's new**: Decomposition technique not typically covered in intro OR courses
- **Why it's better**: Scales to many commodities, only generates "useful" paths

## Data Preparation

### 1. Load and Subsample Network
```python
# Load California road network
# Subsample to ~1000-5000 nodes for experiments
# Use largest connected component
```

### 2. Assign Road Capacities
Options:
- **Uniform**: All roads have same capacity (e.g., 100 vehicles/hour)
- **Degree-based**: Capacity proportional to intersection degrees (major highways have more lanes)
- **Random**: Capacities vary randomly within a range

### 3. Generate Traffic Commodities
Options:
- **Random O-D pairs**: Randomly select origin-destination pairs
- **Geographic patterns**: Long-distance vs short-distance trips
- **Demand variation**: Uniform demands vs heavy-tailed (some routes have much more traffic)

## Computational Experiments

### Variables to Test
1. **Number of commodities** $K$: 5, 10, 20, 50, 100
2. **Network size**: 500, 1000, 2000, 5000 nodes
3. **Capacity tightness**: Vary base capacity to create congestion
4. **Demand patterns**: Uniform vs realistic (distance-based)

### Metrics to Compare
1. **Solution quality**
   - Total vehicles routed (throughput)
   - Percentage of demands satisfied
   - Average travel distance

2. **Computation time**
   - Time vs number of commodities $K$
   - Time vs network size
   - Scalability analysis

3. **Method-specific metrics**
   - Column generation: iterations to converge, paths generated
   - Max flow: satisfaction rate with greedy ordering
   - Optimality gap: % of Gurobi optimal solution

### Expected Results
- **Greedy shortest path**: Very fast (<1s), but 70-85% optimal
- **Sequential max flow**: Fast (~1-5s), 75-90% optimal  
- **Gurobi IP**: Optimal (100%), but slow (10-300s), doesn't scale to large $K$
- **Column generation**: Near-optimal (98-100%), medium speed (2-30s), scales well

## Visualizations

1. **Network visualization**: Show different commodities as colored paths on the road network
2. **Convergence plot**: Column generation objective improving over iterations
3. **Scalability plot**: Time vs $K$ on log scale for all methods
4. **Quality-speed tradeoff**: Scatter plot of solution quality vs computation time
5. **Capacity utilization**: Histogram showing which roads are congested

## Why This Project?

✅ **Clear real-world application**: Traffic routing is intuitive and important  
✅ **Large real dataset**: Actual California road network from SNAP  
✅ **Uses course material**: Shortest path, max flow algorithms  
✅ **Clear new method**: Column generation (decomposition not covered in class)  
✅ **Strong experimental component**: Many parameters to vary, clear metrics  
✅ **Scalability story**: Show when each method wins (greedy vs optimal vs decomposition)  

## Project Requirements Met

- ✅ **Methods from course**: Shortest path (Dijkstra), Max flow (Edmonds-Karp), IP formulation
- ✅ **New method**: Column generation (master-subproblem decomposition)
- ✅ **Computational component**: Real road network data, scalability experiments across $K$ and network size
- ✅ **Optimization focus**: Clear MCF formulation, compare solution approaches

## Expected Timeline

**Days 1-2**: Data loading, baseline implementations (shortest path, max flow)  
**Days 3-4**: Gurobi IP and column generation implementation  
**Days 5-6**: Run experiments, generate results and plots  
**Day 7**: Write report with formulation, methods, and analysis

The Maximum Flow analysis is invaluable in Transportation Engineering and Urban Planning because it offers a definitive measure of network performance and identifies critical failure points.

A. Identifying Critical Bottlenecks (The Min Cut)
The most helpful output, conceptually, is the Minimum Cut. According to the Max Flow, Min Cut Theorem , the maximum flow through a network is always equal to the capacity of the minimum cut (the narrowest passage).

Application: Infrastructure Planning

If a city wants to increase the morning commute throughput between neighborhood A and downtown B, this analysis immediately tells them which specific bridge, tunnel, or junction is the main bottleneck.

It shows that adding lanes to any road outside the Min Cut will not increase the overall maximum flow; all expansion efforts should be concentrated on the minimum cut roads.

B. Disaster and Emergency Response Planning
In emergency scenarios like fire or evacuation, ensuring maximum throughput is critical for saving lives.

Application: Emergency Evacuation

If a coastal city is evacuating, the origin is the city center, and the destination is the nearest highway out. Max Flow analysis determines the maximum possible evacuation rate.

Planners can use the Min Cut to identify which exit ramps or arterial roads must be kept clear, or temporarily converted to one-way traffic, to maximize the outward flow.

C. Network Robustness and Resilience Testing
Maximum Flow helps predict the impact of road closures or infrastructure failure.

Application: Testing Road Closures

By setting the capacity of a critical road segment (like a main bridge) to zero in the model, the analysis can calculate the new, reduced Max Flow.

This instantly quantifies the resulting traffic deficit and helps planners assess network resilience and create effective detour routes with sufficient capacity.

D. Logistics and Supply Chain Management
While the example focuses on personal vehicles, the model applies equally to commodities being shipped.

Application: Delivery Optimization

For a delivery service managing a fleet of trucks, Max Flow analysis helps determine the maximum number of packages (commodity) that can be processed through a distribution hub (source) and reach a delivery zone (sink) within a specific time window, optimizing fleet scheduling and route assignments based on real road capacities.

In summary, the Edmonds-Karp Max Flow analysis is a powerful, foundational tool that moves beyond simple distance measurements to provide a quantifiable, capacity-constrained answer to the question: "How much traffic can our infrastructure truly handle?"